In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "03_fibroblast_gene_expression_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_mouse_annotated.zarr")
adata

In [ ]:
FIBROBLASTS = ["Trophocyte", "Adamdec1+ fibroblast", "Telocyte", "Villus fibroblast"]
CELLS = ["Trophocyte", "Adamdec1+ fibroblast"]
TIME = ["Sham", "24 h", "48 h", "3.5 d"]
RADIATION_COLORS = {"C-ion": fs.CION, "γ-ray": fs.GAMMA}

ROIS = (adata.obs[["roi_no", "radiation", "time", "condition"]].drop_duplicates()
        .astype({"roi_no": int, "radiation": str, "condition": str}).set_index("roi_no").sort_index())
ROIS

In [ ]:
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

IRRADIATED = ROIS[ROIS["time"] != "Sham"].drop_duplicates("condition")
SHAM = ROIS.loc[ROIS["time"] == "Sham"].set_index("radiation")["condition"]

In [ ]:
sub = adata[adata.obs["cell_subtype"].isin(FIBROBLASTS)].copy()
sub.obs["roi_no"] = sub.obs["roi_no"].astype(str)

pdata = dc.pp.pseudobulk(
    adata=sub,
    sample_col="roi_no",
    groups_col="cell_subtype",
    layer="counts",
    mode="sum",
)
dc.pp.filter_samples(pdata, min_cells=10, min_counts=1000)

de_results, normed_counts = [], {}
for cell in FIBROBLASTS:
    tcells = pdata[pdata.obs["cell_subtype"] == cell].copy()
    dc.pp.filter_by_expr(
        adata=tcells,
        group="condition",
        min_count=10,
        min_total_count=15,
        large_n=10,
        min_prop=0.7,
    )
    dc.pp.filter_by_prop(
        adata=tcells,
        min_prop=0.05,
        min_smpls=2,
    )
    for radiation in SHAM.index:
        ref = SHAM[radiation]
        conditions = tcells.obs["condition"].astype(str)
        tcells.obs["condition"] = pd.Categorical(conditions, categories=[ref] + sorted(set(conditions) - {ref}))
        inference = DefaultInference(n_cpus=8)
        dds = DeseqDataSet(
            adata=tcells,
            design="~condition",
            refit_cooks=True,
            inference=inference,
            quiet=True,
        )

        dds.deseq2()

        for condition, time in IRRADIATED.loc[IRRADIATED["radiation"] == radiation, ["condition", "time"]].itertuples(index=False):
            stat_res = DeseqStats(dds, contrast=["condition", condition, ref], inference=inference, quiet=True)

            stat_res.summary()
            stat_res.lfc_shrink(coeff=f"condition[T.{condition}]")
            results_df = stat_res.results_df
            de_results.append(results_df.assign(cell=cell, radiation=radiation, time=time, condition=condition))
    normed_counts[cell] = pd.DataFrame(dds.layers["normed_counts"], index=tcells.obs["roi_no"].astype(int), columns=tcells.var_names.rename("gene"))

In [ ]:
de_results = pd.concat(de_results).rename_axis("gene").reset_index()
de_results.to_csv(OUT_DIR / "fibroblast_deseq2_vs_sham.csv", index=False)
for cell, counts in normed_counts.items():
    counts.to_csv(OUT_DIR / f"fibroblast_normalized_counts_{cell.replace(' ', '_')}.csv")
cols = ["cell_subtype", "roi_no", "radiation", "time", "psbulk_cells", "psbulk_counts"]
pdata.obs[cols].to_csv(OUT_DIR / "fibroblast_pseudobulk_samples.csv", index=False)

In [ ]:
FC_CUTOFF = 0.5

lfc = de_results.pivot_table(index=["cell", "gene"], columns=["radiation", "time"], values="log2FoldChange")
AXIS_LIM = np.nanquantile(lfc, [0.005, 0.995])

GROUPS = {
    "Other": (fs.LIGHTGREY, 2),
    "Up with C-ion only": (fs.CION, 3),
    "Up with γ-ray only": (fs.GAMMA, 3),
    "Up with both": (fs.BOTH, 3),
}

def classify(cell, time):
    x = lfc.loc[cell, ("C-ion", time)]
    y = lfc.loc[cell, ("γ-ray", time)]
    group = pd.Series("Other", index=x.index)
    group[(x >= FC_CUTOFF) & (y < FC_CUTOFF)] = "Up with C-ion only"
    group[(x < FC_CUTOFF) & (y >= FC_CUTOFF)] = "Up with γ-ray only"
    group[(x >= FC_CUTOFF) & (y >= FC_CUTOFF)] = "Up with both"
    return group

def scatter_panel(fig, cell, time):
    ax = fig.subplots()
    x = lfc.loc[cell, ("C-ion", time)]
    y = lfc.loc[cell, ("γ-ray", time)]
    group = classify(cell, time)
    for name, (color, size) in GROUPS.items():
        in_group = group == name
        ax.scatter(x[in_group].clip(AXIS_LIM[0], AXIS_LIM[1]), y[in_group].clip(AXIS_LIM[0], AXIS_LIM[1]),
                   s=size, color=color, lw=0, rasterized=True, label=name)

    ax.plot(AXIS_LIM, AXIS_LIM, color=fs.GREY, lw=0.4, ls="--")
    ax.axvline(FC_CUTOFF, color=fs.GREY, lw=0.4, ls=":")
    ax.axhline(FC_CUTOFF, color=fs.GREY, lw=0.4, ls=":")
    ax.set_xlim(AXIS_LIM)
    ax.set_ylim(AXIS_LIM)
    ax.set_aspect("equal")
    ax.set_title(f"{cell}, {time}")
    ax.set_xlabel("C-ion: log$_2$ fold change vs Sham")
    ax.set_ylabel("γ-ray: log$_2$ fold change vs Sham")

n_genes = {}
for cell in FIBROBLASTS:
    for time in TIME[1:]:
        fig = plt.figure(figsize=(50 * fs.MM, 50 * fs.MM), layout="constrained")
        scatter_panel(fig, cell, time)
        n_genes[(cell, time)] = classify(cell, time).value_counts()
        fs.save(fig, OUT_DIR / "single" / "scatter" / f"scatter_{cell.replace(' ', '_')}_{time.replace(' ', '')}")
        plt.close(fig)

fig = plt.figure(figsize=(fs.COL2, 150 * fs.MM), layout="constrained")
subfigs = fig.subfigures(len(TIME) - 1, len(FIBROBLASTS), squeeze=False)
for i, time in enumerate(TIME[1:]):
    for j, cell in enumerate(FIBROBLASTS):
        scatter_panel(subfigs[i, j], cell, time)
handles, labels = subfigs[0, 0].axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=4, markerscale=3, fontsize=6)
fs.save(fig, OUT_DIR / "fibroblast_time_scatter")
plt.show()

group_counts = pd.DataFrame(n_genes).reindex(list(GROUPS)).fillna(0).astype(int)
group_counts.to_csv(OUT_DIR / "fourway_counts.csv")
group_counts

In [ ]:
SETS = ["Collagens", "ECM Glycoproteins", "ECM Regulators", "ECM-affiliated Proteins", "Secreted Factors"]

net = dc.op.resource("Matrisome", organism="mouse").rename(columns={"subclass": "source", "genesymbol": "target"})
net = net.loc[net["source"].isin(SETS), ["source", "target"]].drop_duplicates()

scores = []
for cell in CELLS:
    log_expr = np.log2(normed_counts[cell].loc[ROIS.index] + 1)
    score, _ = dc.mt.ulm(data=log_expr - log_expr.mean(), net=net, tmin=5)
    score.index = score.index.astype(int)
    scores.append(score.join(ROIS[["radiation", "time"]]).assign(cell=cell))
scores = pd.concat(scores).rename_axis("roi_no").reset_index()
scores.to_csv(OUT_DIR / "matrisome_scores.csv", index=False)

mean_scores = scores.groupby(["cell", "radiation", "time"])[SETS].mean()
vmax = np.abs(mean_scores.values).max()
columns = [(radiation, time) for radiation in RADIATION_COLORS for time in TIME]

for cell in CELLS:
    fig, ax = plt.subplots(figsize=(80 * fs.MM, 50 * fs.MM), layout="constrained")
    im = ax.imshow(mean_scores.loc[cell].loc[columns].T, cmap="PuOr_r", vmin=-vmax, vmax=vmax, aspect="auto", interpolation="none")
    ax.set_yticks(range(len(SETS)), SETS)
    ax.set_xticks(range(len(columns)), [f"{radiation} {time}" for radiation, time in columns], rotation=90)
    for label, (radiation, time) in zip(ax.get_xticklabels(), columns):
        label.set_color(RADIATION_COLORS[radiation])
    ax.axvline(len(TIME) - 0.5, color=fs.GREY, lw=0.4)
    ax.set_title(cell)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Score (ULM, mean of ROIs)")
    fs.save(fig, OUT_DIR / f"matrisome_heatmap_{cell.replace(' ', '_')}")

In [ ]:
GENES = ["Cdkn1a", "Socs3", "Stat1", "Cxcl9", "Ccl2", "Nrg1", "Ereg", "Igf1"]
GENES_ECM = ["Col1a1", "Col1a2", "Fn1", "Lox", "Mmp2", "Mmp3", "Timp1"]
GENES_TXT = ["Mmp12", "Mmp13", "Ccl12", "Cxcl13"]

expr_matrix = adata.layers["log1p_norm"].tocsc()
PALETTE = {"C-ion, 1st ROI": fs.CION, "C-ion, 2nd ROI": fs.SKY, "γ-ray, 1st ROI": fs.GAMMA, "γ-ray, 2nd ROI": fs.ORANGE}
obs = adata.obs.copy()
obs["group"] = obs["radiation"].astype(str) + ", " + obs["slide"].astype(str).map({"No1": "1st ROI", "No2": "2nd ROI"})

def strip_panel(fig, gene, cell):
    ax = fig.subplots()
    np.random.seed(0)
    is_cell = (obs["cell_subtype"] == cell).values
    expr = expr_matrix[:, adata.var_names.get_loc(gene)].toarray().ravel()
    plot_df = obs.loc[is_cell, ["time", "group"]].copy()
    plot_df["expr"] = expr[is_cell]
    plot_df = plot_df[plot_df["expr"] > 0]
    sns.stripplot(
        data=plot_df, x="time", y="expr", hue="group", order=TIME, hue_order=list(PALETTE), palette=PALETTE,
        dodge=True, size=1, linewidth=0, zorder=1, legend=False, rasterized=True, ax=ax,
    )
    sns.pointplot(
        data=plot_df, x="time", y="expr", hue="group", order=TIME, hue_order=list(PALETTE),
        dodge=.8 - .8 / 4, palette=[fs.BLACK] * 4, errorbar=None, estimator="median",
        markers="_", markersize=6, markeredgewidth=0.8, linestyle="none", legend=False, ax=ax,
    )
    ax.set_title(fs.gene(gene))
    ax.set_xlabel("")
    ax.set_ylabel("Gene expression")

for gene in GENES + GENES_ECM + GENES_TXT:
    for cell in CELLS:
        fig = plt.figure(figsize=(48 * fs.MM, 40 * fs.MM), layout="constrained")
        strip_panel(fig, gene, cell)
        fs.save(fig, OUT_DIR / "single" / "strip" / f"strip_{gene}_{cell.replace(' ', '_')}")
        plt.close(fig)

for genes, name in [(GENES, "fibroblast_time_strips"), (GENES_ECM, "fibroblast_time_strips_ecm"), (GENES_TXT, "fibroblast_time_strips_results")]:
    fig = plt.figure(figsize=((10 + 48 * len(CELLS)) * fs.MM, (10 + 40 * len(genes)) * fs.MM), layout="constrained")
    subfigs = fig.subfigures(len(genes), len(CELLS), squeeze=False)
    for i, gene in enumerate(genes):
        for j, cell in enumerate(CELLS):
            strip_panel(subfigs[i, j], gene, cell)
            if i == 0:
                subfigs[i, j].suptitle(cell, fontsize=7)
    legend = [Line2D([], [], marker="o", ls="", ms=3, color=color, label=label) for label, color in PALETTE.items()]
    fig.legend(handles=legend, loc="outside upper center", ncol=4, fontsize=6)
    fs.save(fig, OUT_DIR / name)
    plt.show()